[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jdsanch1/simrc/blob/master/01.%20Parte%201/08.%20Clase%208/08Class%20NB.ipynb)
[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/jdsanch1/SimRC/master?labpath=01.%20Parte%201%2F08.%20Clase%208%2F08Class%20NB.ipynb)

In [ ]:
import importlib, subprocess, sys

_required = ["yfinance", "pandas", "numpy", "matplotlib", "seaborn", "scipy", "sklearn", "statsmodels"]
_missing  = [pkg for pkg in _required if importlib.util.find_spec(pkg) is None]
if _missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + _missing)

# Clase 8: Resumen Parte 1 — Valuación de opciones por simulación Monte Carlo

**[Juan Diego Sánchez Torres](https://www.researchgate.net/profile/Juan_Diego_Sanchez_Torres)**  
*Profesor*, [MAF ITESO](http://maf.iteso.mx/web/general/detalle?group_id=5858156)  
dsanchez@iteso.mx

---

## Introducción

Esta clase consolida los conceptos de la Parte 1 aplicándolos a la **valuación de opciones por simulación Monte Carlo**. Combinamos:

- **Clase 1–2**: Descarga de datos, rendimientos logarítmicos
- **Clase 3**: Opciones — payoff, P&L, Black-Scholes
- **Clase 5–6**: Estimadores robustos (Shrunk Covariance, Huber)
- **Clase 7**: Simulación Monte Carlo y medidas de riesgo

### Valuación por Monte Carlo

El precio de una opción europea es el valor esperado descontado de su payoff bajo la medida de riesgo neutro $\mathbb{Q}$:

$$
C_0 = e^{-rT} \, \mathbb{E}^{\mathbb{Q}}[\max(S_T - K, 0)]
$$

$$
P_0 = e^{-rT} \, \mathbb{E}^{\mathbb{Q}}[\max(K - S_T, 0)]
$$

Por Monte Carlo, se aproxima generando $N$ trayectorias de $S_T$ y promediando:

$$
\hat{C}_0 = e^{-rT} \frac{1}{N} \sum_{i=1}^{N} \max(S_T^{(i)} - K, 0)
$$

La calidad de la aproximación depende del **modelo de rendimientos** usado para simular $S_T$ (Glasserman, 2003; Venegas Martínez, 2008, Cap. 7).

## 1. Datos y rendimientos

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
from sklearn.neighbors import KernelDensity

sns.set_theme(style="darkgrid", palette="tab10")
plt.rcParams.update({"figure.dpi": 100, "figure.figsize": (12, 5)})
pd.set_option("display.precision", 4)

In [ ]:
ticker = "AAPL"
data = yf.download(ticker, start="2025-01-01", end="2025-03-27",
                   auto_adjust=True, progress=False)
closes = data["Close"].dropna()
daily_returns = np.log(closes / closes.shift(1)).dropna()

S0 = closes.iloc[-1]
mu = daily_returns.mean()
sigma = daily_returns.std()

print(f"Precio actual S₀ = ${S0:.2f}")
print(f"μ diario = {mu:.6f},  σ diario = {sigma:.6f}")
print(f"μ anualizado ≈ {mu*252:.4f},  σ anualizado ≈ {sigma*np.sqrt(252):.4f}")

---
## 2. Modelo 1: Valuación con rendimientos normales

Bajo el supuesto GBM, el precio a $T$ días:

$$
S_T = S_0 \exp\!\left(\sum_{t=1}^{T} r_t\right), \qquad r_t \sim \mathcal{N}(\mu, \sigma^2)
$$

Para la valuación bajo riesgo neutro, reemplazamos $\mu$ por $r_f - \sigma^2/2$ (el drift risk-neutral).

In [ ]:
np.random.seed(42)
K = 170          # Strike
T = 180          # Días al vencimiento
N_traj = 10000   # Trayectorias
r_f = 0.045 / 252  # Tasa libre de riesgo diaria (~4.5% anual)

# Drift risk-neutral
drift_rn = r_f - 0.5 * sigma**2

# Simular trayectorias
sim_returns = np.random.normal(drift_rn, sigma, size=(T, N_traj))
sim_prices = S0 * np.exp(np.cumsum(sim_returns, axis=0))
ST = sim_prices[-1]  # Precios finales

# Valuación
discount = np.exp(-r_f * T)
call_payoffs = np.maximum(ST - K, 0)
put_payoffs = np.maximum(K - ST, 0)
call_price = discount * call_payoffs.mean()
put_price = discount * put_payoffs.mean()

# Error estándar Monte Carlo
call_se = discount * call_payoffs.std() / np.sqrt(N_traj)
put_se = discount * put_payoffs.std() / np.sqrt(N_traj)

print(f"Strike K = ${K}")
print(f"\nModelo Normal ({N_traj:,} trayectorias, {T} días):")
print(f"  Call: ${call_price:.4f} ± ${call_se:.4f}")
print(f"  Put:  ${put_price:.4f} ± ${put_se:.4f}")

In [ ]:
# Visualizar trayectorias y distribución final
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Trayectorias (mostrar 200)
dates_future = pd.date_range(closes.index[-1] + pd.Timedelta(days=1), periods=T, freq="B")
axes[0].plot(dates_future, sim_prices[:, :200], alpha=0.05, color="steelblue")
axes[0].axhline(K, color="red", linestyle="--", label=f"Strike K = {K}")
axes[0].set_title(f"Trayectorias simuladas — Normal")
axes[0].set_ylabel("Precio (USD)")
axes[0].legend()

# Distribución de ST
sns.histplot(ST, bins=80, stat="density", ax=axes[1], alpha=0.6)
axes[1].axvline(K, color="red", linestyle="--", label=f"K = {K}")
axes[1].axvline(ST.mean(), color="green", linestyle="--", label=f"E[S_T] = {ST.mean():.0f}")
axes[1].set_title("Distribución de precio final S_T")
axes[1].set_xlabel("Precio (USD)")
axes[1].legend()
plt.tight_layout()

---
## 3. Modelo 2: Valuación con histograma empírico

Muestreamos de la distribución empírica de rendimientos, capturando la forma real (colas pesadas, asimetría) sin imponer normalidad.

In [ ]:
# Histograma empírico
values, bin_edges = np.histogram(daily_returns, bins=200)
weights = values / values.sum()
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

sim_returns_hist = np.random.choice(bin_centers, size=(T, N_traj), p=weights)
sim_prices_hist = S0 * np.exp(np.cumsum(sim_returns_hist, axis=0))
ST_hist = sim_prices_hist[-1]

call_hist = discount * np.maximum(ST_hist - K, 0).mean()
put_hist = discount * np.maximum(K - ST_hist, 0).mean()
call_hist_se = discount * np.maximum(ST_hist - K, 0).std() / np.sqrt(N_traj)
put_hist_se = discount * np.maximum(K - ST_hist, 0).std() / np.sqrt(N_traj)

print(f"Modelo Histograma ({N_traj:,} trayectorias):")
print(f"  Call: ${call_hist:.4f} ± ${call_hist_se:.4f}")
print(f"  Put:  ${put_hist:.4f} ± ${put_hist_se:.4f}")

---
## 4. Modelo 3: Valuación con KDE

La estimación de densidad por kernel suaviza la distribución empírica, produciendo muestras continuas que preservan las características de los datos reales.

In [ ]:
kde = KernelDensity(kernel="gaussian", bandwidth=0.002).fit(
    daily_returns.values.reshape(-1, 1))

sim_returns_kde = kde.sample(n_samples=T * N_traj, random_state=42).reshape(T, N_traj)
sim_prices_kde = S0 * np.exp(np.cumsum(sim_returns_kde, axis=0))
ST_kde = sim_prices_kde[-1]

call_kde = discount * np.maximum(ST_kde - K, 0).mean()
put_kde = discount * np.maximum(K - ST_kde, 0).mean()
call_kde_se = discount * np.maximum(ST_kde - K, 0).std() / np.sqrt(N_traj)
put_kde_se = discount * np.maximum(K - ST_kde, 0).std() / np.sqrt(N_traj)

print(f"Modelo KDE ({N_traj:,} trayectorias):")
print(f"  Call: ${call_kde:.4f} ± ${call_kde_se:.4f}")
print(f"  Put:  ${put_kde:.4f} ± ${put_kde_se:.4f}")

---
## 5. Comparación de los tres modelos

### Precios estimados

In [ ]:
comparison = pd.DataFrame({
    "Modelo": ["Normal", "Histograma", "KDE"],
    "Call": [call_price, call_hist, call_kde],
    "Call SE": [call_se, call_hist_se, call_kde_se],
    "Put": [put_price, put_hist, put_kde],
    "Put SE": [put_se, put_hist_se, put_kde_se],
    "E[S_T]": [ST.mean(), ST_hist.mean(), ST_kde.mean()],
    "Std[S_T]": [ST.std(), ST_hist.std(), ST_kde.std()],
}).set_index("Modelo")
comparison

In [ ]:
# Distribuciones de S_T superpuestas
fig, ax = plt.subplots(figsize=(12, 6))
for prices, label, color in [
    (ST, "Normal", "steelblue"),
    (ST_hist, "Histograma", "tab:orange"),
    (ST_kde, "KDE", "tab:green"),
]:
    sns.histplot(prices, bins=80, stat="density", alpha=0.3, label=label, color=color, ax=ax)
ax.axvline(K, color="red", linestyle="--", linewidth=2, label=f"K = {K}")
ax.set_title(f"Distribución de S_T por modelo ({N_traj:,} trayectorias, T = {T} días)")
ax.set_xlabel("Precio final (USD)")
ax.legend()
plt.tight_layout()

---
## 6. Evolución del precio de la opción en el tiempo

El valor de la opción varía con el tiempo al vencimiento. Calculamos el precio de la call y la put para cada horizonte intermedio $t = 1, \ldots, T$.

In [ ]:
# Precio de call y put para cada horizonte t
call_evolution = np.zeros(T)
put_evolution = np.zeros(T)

for t in range(T):
    ST_t = sim_prices[t]
    disc_t = np.exp(-r_f * (t + 1))
    call_evolution[t] = disc_t * np.maximum(ST_t - K, 0).mean()
    put_evolution[t] = disc_t * np.maximum(K - ST_t, 0).mean()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(range(1, T+1), call_evolution, label="Call", linewidth=2)
ax.plot(range(1, T+1), put_evolution, label="Put", linewidth=2)
ax.set_xlabel("Días al vencimiento")
ax.set_ylabel("Prima estimada (USD)")
ax.set_title(f"Evolución del precio de opciones — Modelo Normal (K = {K})")
ax.legend()
plt.tight_layout()

---
## 7. Convergencia Monte Carlo

### Error estándar

El error estándar de la estimación Monte Carlo decrece como $1/\sqrt{N}$:

$$
\text{SE} = \frac{\hat{\sigma}_{\text{payoff}}}{\sqrt{N}}
$$

Verificamos la convergencia aumentando el número de trayectorias:

In [ ]:
n_values = [100, 500, 1000, 5000, 10000, 50000]
call_estimates = []

for n in n_values:
    sim_r = np.random.normal(drift_rn, sigma, size=(T, n))
    sim_p = S0 * np.exp(np.cumsum(sim_r, axis=0))
    payoff = np.maximum(sim_p[-1] - K, 0)
    est = discount * payoff.mean()
    se = discount * payoff.std() / np.sqrt(n)
    call_estimates.append({"N": n, "Precio": est, "SE": se})

conv_df = pd.DataFrame(call_estimates)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].errorbar(conv_df["N"], conv_df["Precio"], yerr=2*conv_df["SE"],
                 fmt="o-", capsize=4, color="steelblue")
axes[0].set_xscale("log")
axes[0].set_xlabel("Número de trayectorias (N)")
axes[0].set_ylabel("Precio estimado (USD)")
axes[0].set_title("Convergencia del precio de la Call")

axes[1].plot(conv_df["N"], conv_df["SE"], "o-", color="tab:red")
axes[1].set_xscale("log")
axes[1].set_yscale("log")
axes[1].set_xlabel("Número de trayectorias (N)")
axes[1].set_ylabel("Error estándar (USD)")
axes[1].set_title("Error estándar vs. N (pendiente ≈ -0.5)")
plt.tight_layout()

---
## 8. Resumen de la Parte 1

| Clase | Tema | Herramienta clave |
|-------|------|-------------------|
| 1 | Análisis de acciones | `yfinance`, rendimientos log, volatilidad |
| 2 | Retornos y covarianza | Matriz Σ, correlación, regresión |
| 3 | Opciones financieras | Payoff, P&L, volatilidad implícita |
| 4 | Portafolio óptimo | Sharpe, frontera eficiente, CVXPY (DCP) |
| 5 | Covarianza robusta | `ShrunkCovariance`, `LedoitWolf` |
| 6 | Media robusta + RNG | Huber, LCG, Box-Muller |
| 7 | Simulación de riesgo | Monte Carlo, VaR, CVaR |
| **8** | **Valuación MC de opciones** | **Normal, histograma, KDE** |

### Flujo completo

```
Datos → Rendimientos → Estimadores (robustos) → Simulación MC → Valuación / Riesgo
```

En la **Parte 2** se profundiza en: optimización avanzada con CVXPY, inclusión de bonos, opciones barrera y programación estocástica con Pyomo.

---

## 9. Referencias bibliográficas

- **Black, F. & Scholes, M.** (1973). The Pricing of Options and Corporate Liabilities. *Journal of Political Economy*, 81(3), 637–654.
- **Boyle, P. P.** (1977). Options: A Monte Carlo Approach. *Journal of Financial Economics*, 4(3), 323–338.
- **Cont, R.** (2001). Empirical Properties of Asset Returns. *Quantitative Finance*, 1(2), 223–236.
- **Glasserman, P.** (2003). *Monte Carlo Methods in Financial Engineering*. Springer. — Cap. 1–4.
- **Hull, J. C.** (2018). *Options, Futures, and Other Derivatives* (10th ed.). Pearson. — Cap. 15, 22.
- **Ledoit, O. & Wolf, M.** (2004). A well-conditioned estimator for large-dimensional covariance matrices. *Journal of Multivariate Analysis*, 88(2), 365–411.
- **Luenberger, D. G.** (2013). *Investment Science* (2nd ed.). Oxford University Press.
- **McNeil, A. J., Frey, R. & Embrechts, P.** (2015). *Quantitative Risk Management* (2nd ed.). Princeton University Press.
- **Tsay, R. S.** (2010). *Analysis of Financial Time Series* (3rd ed.). Wiley.
- **Venegas Martínez, F.** (2008). *Riesgos financieros y económicos* (2a ed.). Cengage Learning. — Cap. 7: Simulación Monte Carlo.